# Fix Metadata Evaluation

This notebook runs evaluations of the metadata correction pipeline and organizes the results in a structured way.

## Setup and Configuration

In [1]:
import os
import sys
import json
import glob
import asyncio
from datetime import datetime
from pathlib import Path
import pandas as pd
from loguru import logger

# Add fix_values package to path
# sys.path.append("../../../lambdas/fix_metadata/fix_metadata/fix_values/src")

from fix_values.evaluation import (
    PipelineEvaluator,
    EvaluationResult
)
from fix_values.pipeline import (
    MetadataCorrectionConfig,
    PipelineSettings,
    ValidationSettings,
    SemanticSettings
)
from fix_values.evaluation.visualizations import (
    plot_stage_metrics,
    plot_field_metrics,
    plot_evaluation_metrics,
    save_visualizations
)

ModuleNotFoundError: No module named 'plotly'

In [ ]:
class EvaluationConfig:
    """Configuration for an evaluation run."""
    
    def __init__(self, run_name: str):
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.run_name = run_name
        self.run_dir = f"evaluation_runs/{self.timestamp}_{run_name}"
        
        # Directory paths
        self.config_dir = f"{self.run_dir}/config"
        self.logs_dir = f"{self.run_dir}/logs"
        self.results_dir = f"{self.run_dir}/results"
        self.viz_dir = f"{self.run_dir}/visualizations"
        
        # File paths
        self.schema_path = "/home/ubuntu/projects/data/pz-nf-testing-data/schema/NF.jsonid"
        self.input_dir = "/home/ubuntu/projects/data/synthetic_error_limited"
        self.ground_truth_dir = "/home/ubuntu/projects/data/ground_truth"
        self.log_file = f"{self.logs_dir}/pipeline_results.log"
        self.results_file = f"{self.results_dir}/results_batch.json"
        
    def create_directories(self):
        """Create directory structure for the run."""
        for dir_path in [self.config_dir, self.logs_dir, self.results_dir, self.viz_dir]:
            os.makedirs(dir_path, exist_ok=True)
    
    def to_dict(self):
        """Convert config to dictionary for saving."""
        return {
            "timestamp": self.timestamp,
            "run_name": self.run_name,
            "schema_path": self.schema_path,
            "input_dir": self.input_dir,
            "ground_truth_dir": self.ground_truth_dir,
            "run_dir": self.run_dir,
            "config_dir": self.config_dir,
            "logs_dir": self.logs_dir,
            "results_dir": self.results_dir,
            "viz_dir": self.viz_dir,
            "log_file": self.log_file,
            "results_file": self.results_file
        }
    
    def save_json(self, filepath: str):
        """Save configuration to JSON file."""
        with open(filepath, 'w') as f:
            json.dump(self.to_dict(), f, indent=2)

## Run Evaluation

In [ ]:
async def run_evaluation(eval_config: EvaluationConfig):
    """Run evaluation pipeline and generate visualizations."""
    # Create directories
    eval_config.create_directories()
    
    # Configure logging
    logger.remove()  # Remove default handlers
    
    # Add console handler for progress only
    logger.add(
        sys.stdout,
        level="INFO",
        format="{message}",
        filter=lambda record: "Progress:" in str(record["message"]) or "Running" in str(record["message"]) or "complete" in str(record["message"])
    )
    
    # Add file handler for all logs
    logger.add(
        eval_config.log_file,
        rotation="100 MB",
        level="DEBUG",
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}"
    )
    
    # Save configuration
    eval_config.save_json(f"{eval_config.config_dir}/evaluation_config.json")
    
    # Create pipeline configuration
    pipeline_config = MetadataCorrectionConfig(
        pipeline=PipelineSettings(
            schema_path=eval_config.schema_path,
            output_dir=eval_config.results_dir,
            log_level="DEBUG"
        )
    )
    
    # Initialize evaluator
    evaluator = PipelineEvaluator(pipeline_config)
    
    print("Running evaluation...")
    results = await evaluator.evaluate_directory(
        input_dir=eval_config.input_dir,
        ground_truth_dir=eval_config.ground_truth_dir
    )
    
    # Save results
    evaluator.save_results(results, eval_config.results_file)
    
    print("\nGenerating visualizations...")
    # Load results to get BatchEvaluationResult
    with open(eval_config.results_file) as f:
        batch_results = json.load(f)
    
    # Generate stage metrics plots
    stage_figs = plot_stage_metrics(batch_results["overall_metrics"]["stage_metrics"])
    save_visualizations(stage_figs, eval_config.viz_dir, "stage")
    
    # Generate field metrics plots
    field_figs = plot_field_metrics(batch_results["overall_metrics"]["field_metrics"])
    save_visualizations(field_figs, eval_config.viz_dir, "field")
    
    # Generate overall metrics plots
    eval_figs = plot_evaluation_metrics(batch_results["overall_metrics"])
    save_visualizations(eval_figs, eval_config.viz_dir, "overall")
    
    print("\nEvaluation complete! Results saved to:")
    print(f"Config: {eval_config.config_dir}/evaluation_config.json")
    print(f"Results: {eval_config.results_file}")
    print(f"Visualizations: {eval_config.viz_dir}")
    print(f"Logs: {eval_config.log_file}")
    
    return batch_results

In [ ]:
# Initialize and run evaluation
config = EvaluationConfig("bedrock_titan_v1")
results = await run_evaluation(config)

Running evaluation...


/home/ubuntu/projects/genai-curator/client/demos/fix_metadata/../../../lambdas/fix_metadata/fix_metadata/fix_values/src/fix_values/pipeline/nodes/validation.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  state.metadata.at[idx, field.name] = converted_value
/home/ubuntu/projects/genai-curator/client/demos/fix_metadata/../../../lambdas/fix_metadata/fix_metadata/fix_values/src/fix_values/pipeline/nodes/validation.py:226: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  state.metadata.at[idx, field.name] = converted_value


[initial_validation] Validation complete. Found 3480 total errors:
[Node: initial_validation] Execution completed in 0.496s
[Node: fuzzy_match] Execution completed in 0.004s
[post_fuzzy_validation] Validation complete. Found 3480 total errors:
[Node: post_fuzzy_validation] Execution completed in 0.529s
[Node: semantic_similarity] Execution completed in 0.512s
[post_semantic_validation] Validation complete. Found 3480 total errors:
[Node: post_semantic_validation] Execution completed in 0.625s
[Node: inference] Execution completed in 0.776s
[post_inference_validation] Validation complete. Found 3480 total errors:
[Node: post_inference_validation] Execution completed in 0.603s
[Node: llm_correction] Execution completed in 88.176s
[final_validation] Validation complete. Found 2164 total errors:
[Node: final_validation] Execution completed in 0.765s


/home/ubuntu/projects/genai-curator/client/demos/fix_metadata/../../../lambdas/fix_metadata/fix_metadata/fix_values/src/fix_values/evaluation/evaluator.py:112: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  error_df.at[error.row_idx, error.column] = True
/home/ubuntu/projects/genai-curator/client/demos/fix_metadata/../../../lambdas/fix_metadata/fix_metadata/fix_values/src/fix_values/evaluation/metrics.py:62: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  stage_mask[corrected_indices] = True


Progress: 1/1 files processed (100.0%)
Batch evaluation complete. Successfully processed 1/1 files

Generating visualizations...


/home/ubuntu/miniconda3/envs/py313/lib/python3.13/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(3638), input_type=int64])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(114), input_type=int64])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(76), input_type=int64])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(120), input_type=int64])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(0), input_type=int64])
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=np.int64(76), input_type=int64])
  PydanticSerial


Evaluation complete! Results saved to:
Config: evaluation_runs/20250729_205457_bedrock_titan_v1/config/evaluation_config.json
Results: evaluation_runs/20250729_205457_bedrock_titan_v1/results/results_batch.json
Visualizations: evaluation_runs/20250729_205457_bedrock_titan_v1/visualizations
Logs: evaluation_runs/20250729_205457_bedrock_titan_v1/logs/pipeline_results.log


## View Results

In [ ]:
print("Overall Metrics:")
print(json.dumps(results["overall_metrics"], indent=2))

Overall Metrics:
{
  "total_files": 1,
  "base_accuracy": 0.04875,
  "total_corrections": 3638,
  "total_errors": 4566,
  "correction_accuracy": 0.009620670698185816,
  "correction_rate": 0.7967586508979413,
  "total_execution_time": 92.68992829322815,
  "files_with_errors": [
    "/home/ubuntu/projects/data/metadata_extract_source/nf_8_with_errors.csv"
  ],
  "stage_metrics": {
    "CorrectionMethod.FUZZY": {
      "corrections": 0.0,
      "correction_accuracy": 0.0,
      "correction_rate": 0.0
    },
    "CorrectionMethod.SEMANTIC": {
      "corrections": 0.0,
      "correction_accuracy": 0.0,
      "correction_rate": 0.0
    },
    "CorrectionMethod.INFERENCE": {
      "corrections": 0.0,
      "correction_accuracy": 0.0,
      "correction_rate": 0.0
    },
    "CorrectionMethod.LLM": {
      "corrections": 98.0,
      "correction_accuracy": 0.35714285714285715,
      "correction_rate": 0.02146298729741568
    },
    "CorrectionMethod.MANUAL": {
      "corrections": 0.0,
      "co

## Run History

In [ ]:
def list_evaluation_runs() -> pd.DataFrame:
    """List all evaluation runs with their configurations and results."""
    runs = []
    for run_dir in sorted(glob.glob("evaluation_runs/*")):
        try:
            config_file = f"{run_dir}/config/evaluation_config.json"
            results_file = f"{run_dir}/results/results_batch.json"
            
            with open(config_file) as f:
                config = json.load(f)
            with open(results_file) as f:
                results = json.load(f)
                
            runs.append({
                "timestamp": config["timestamp"],
                "name": config["run_name"],
                "metrics": results["overall_metrics"],
                "artifacts": {
                    "config": config_file,
                    "results": results_file,
                    "visualizations": f"{run_dir}/visualizations/"
                }
            })
        except Exception as e:
            print(f"Error loading run {run_dir}: {str(e)}")
            continue
            
    return pd.DataFrame(runs)

# List all evaluation runs
runs_df = list_evaluation_runs()
display(runs_df)

,timestamp,name,metrics,artifacts
0,20250729_153801,bedrock_titan_v1,"{'total_files': 1, 'base_accuracy': 0.04875, '...",{'config': 'evaluation_runs/20250729_153801_be...
1,20250729_180021,bedrock_titan_v1,"{'total_files': 1, 'base_accuracy': 0.04875, '...",{'config': 'evaluation_runs/20250729_180021_be...
2,20250729_180316,bedrock_titan_v1,"{'total_files': 5, 'base_accuracy': 0.68760515...",{'config': 'evaluation_runs/20250729_180316_be...
3,20250729_190040,bedrock_titan_v1,"{'total_files': 0, 'base_accuracy': 0.0, 'tota...",{'config': 'evaluation_runs/20250729_190040_be...
4,20250729_190342,bedrock_titan_v1,"{'total_files': 0, 'base_accuracy': 0.0, 'tota...",{'config': 'evaluation_runs/20250729_190342_be...
5,20250729_191321,bedrock_titan_v1,"{'total_files': 0, 'base_accuracy': 0.0, 'tota...",{'config': 'evaluation_runs/20250729_191321_be...
6,20250729_191600,bedrock_titan_v1,"{'total_files': 5, 'base_accuracy': 0.68760515...",{'config': 'evaluation_runs/20250729_191600_be...
7,20250729_192842,bedrock_titan_v1,"{'total_files': 5, 'base_accuracy': 0.68760515...",{'config': 'evaluation_runs/20250729_192842_be...
8,20250729_194427,bedrock_titan_v1,"{'total_files': 1, 'base_accuracy': 0.04875, '...",{'config': 'evaluation_runs/20250729_194427_be...
9,20250729_203044,bedrock_titan_v1,"{'total_files': 1, 'base_accuracy': 0.04875, '...",{'config': 'evaluation_runs/20250729_203044_be...
